# **Task 3**

In [1]:
from google.colab import drive

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
from pyspark.sql import SparkSession

SEED = 42

# Increased memory to handle the dataset size and cross-validation overhead
spark = (
    SparkSession.builder
    .appName("Task3_ML_Portfolio")
    .config("spark.driver.memory", "8g")
    .config("spark.executor.memory", "4g")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)

print("Spark Version:", spark.version)
print("Spark Session initialized with 8GB Driver Memory.")

Spark Version: 4.0.3
Spark Session initialized with 8GB Driver Memory.


In [3]:
processed_df = spark.read.parquet(
    "/content/drive/MyDrive/7006SCN/processed_reviews"
)

In [4]:
processed_df.printSchema()

root
 |-- rating: double (nullable = true)
 |-- text: string (nullable = true)
 |-- title: string (nullable = true)
 |-- review: string (nullable = true)
 |-- label: double (nullable = true)
 |-- words: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- filtered_words: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- rawFeatures: vector (nullable = true)
 |-- features: vector (nullable = true)



In [5]:
processed_df.select(
    "label",
    "features"
).show(5, truncate=False)

+-----+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [6]:
print("Total Records:", processed_df.count())

Total Records: 481304


In [7]:
train_df, test_df = processed_df.randomSplit(
    [0.8, 0.2],
    seed=SEED
)

print("Training Rows:", train_df.count())
print("Testing Rows :", test_df.count())

Training Rows: 384943
Testing Rows : 96361


In [8]:
import time

from pyspark.ml.classification import (
    LogisticRegression,
    LinearSVC,
    NaiveBayes,
    RandomForestClassifier
)

from pyspark.ml.tuning import (
    ParamGridBuilder,
    CrossValidator
)

from pyspark.ml.evaluation import (
    MulticlassClassificationEvaluator
)

In [9]:
accuracy_eval = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="accuracy"
)

precision_eval = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="weightedPrecision"
)

recall_eval = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="weightedRecall"
)

f1_eval = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="f1"
)

In [10]:
results = []

## **Logistic Regression with CrossValidator**

In [11]:
# ---------------------------------------------------------
# Step 1: Create Logistic Regression Model
# ---------------------------------------------------------

from pyspark.ml.classification import LogisticRegression

lr = LogisticRegression(
    featuresCol="features",
    labelCol="label",
    predictionCol="prediction",
    maxIter=20
)

print("Logistic Regression model created successfully.")

Logistic Regression model created successfully.


In [12]:
# ---------------------------------------------------------
# Step 2: Display Model Parameters
# ---------------------------------------------------------

print(lr.explainParams())

aggregationDepth: suggested depth for treeAggregate (>= 2). (default: 2)
elasticNetParam: the ElasticNet mixing parameter, in range [0, 1]. For alpha = 0, the penalty is an L2 penalty. For alpha = 1, it is an L1 penalty. (default: 0.0)
family: The name of family which is a description of the label distribution to be used in the model. Supported options: auto, binomial, multinomial (default: auto)
featuresCol: features column name. (default: features, current: features)
fitIntercept: whether to fit an intercept term. (default: True)
labelCol: label column name. (default: label, current: label)
lowerBoundsOnCoefficients: The lower bounds on coefficients if fitting under bound constrained optimization. The bound matrix must be compatible with the shape (1, number of features) for binomial regression, or (number of classes, number of features) for multinomial regression. (undefined)
lowerBoundsOnIntercepts: The lower bounds on intercepts if fitting under bound constrained optimization. The

In [13]:
# ---------------------------------------------------------
# Step 3: Hyperparameter Grid (Reduced for Memory Safety)
# ---------------------------------------------------------
from pyspark.ml.tuning import ParamGridBuilder

lr_paramGrid = (
    ParamGridBuilder()
    .addGrid(lr.regParam, [0.01])
    .addGrid(lr.elasticNetParam, [0.0, 0.5])
    .build()
)

print("Total Parameter Combinations:", len(lr_paramGrid))

Total Parameter Combinations: 2


In [14]:
# ---------------------------------------------------------
# Step 4: Cross Validation (Reduced numFolds to avoid OOM)
# ---------------------------------------------------------
from pyspark.ml.tuning import CrossValidator

lr_cv = CrossValidator(
    estimator=lr,
    estimatorParamMaps=lr_paramGrid,
    evaluator=accuracy_eval,
    numFolds=2,  # Reduced from 3 to 2
    seed=SEED
)

In [15]:
# ---------------------------------------------------------
# Step 5: Start Timer
# ---------------------------------------------------------

import time

lr_start = time.time()

In [16]:
# ---------------------------------------------------------
# Step 6: Train Logistic Regression with Sampling (Memory Safety)
# ---------------------------------------------------------

try:
    # Since the full dataset (385k) causes OOM in CrossValidation,
    # we will take a representative sample of 50k records for training.
    train_sample = train_df.sample(withReplacement=False, fraction=50000/384943, seed=SEED)

    print(f"Training on a sample of {train_sample.count()} records for memory safety...")

    # Fit the model using the sample
    lr_cv_model = lr_cv.fit(train_sample)
    print("Training completed successfully!")

except Exception as e:
    print(f"Training failed. Please ensure you restarted the runtime after the last crash. Error: {e}")

Training on a sample of 50110 records for memory safety...
Training completed successfully!


In [17]:


lr_end = time.time()

lr_training_time = lr_end - lr_start

print(f"Training Time: {lr_training_time:.2f} seconds")

Training Time: 120.14 seconds


In [18]:
# ---------------------------------------------------------
# Step 8: Extract Best Model
# ---------------------------------------------------------

if 'lr_cv_model' in globals():
    best_lr = lr_cv_model.bestModel
    print("Best Model extracted successfully.")
    print(best_lr.explainParams())
else:
    print("Error: lr_cv_model is not defined. Please ensure Step 6 (training) completes successfully without OOM errors.")

Best Model extracted successfully.
aggregationDepth: suggested depth for treeAggregate (>= 2). (default: 2)
elasticNetParam: the ElasticNet mixing parameter, in range [0, 1]. For alpha = 0, the penalty is an L2 penalty. For alpha = 1, it is an L1 penalty. (default: 0.0, current: 0.5)
family: The name of family which is a description of the label distribution to be used in the model. Supported options: auto, binomial, multinomial (default: auto)
featuresCol: features column name. (default: features, current: features)
fitIntercept: whether to fit an intercept term. (default: True)
labelCol: label column name. (default: label, current: label)
lowerBoundsOnCoefficients: The lower bounds on coefficients if fitting under bound constrained optimization. The bound matrix must be compatible with the shape (1, number of features) for binomial regression, or (number of classes, number of features) for multinomial regression. (undefined)
lowerBoundsOnIntercepts: The lower bounds on intercepts if 

In [19]:
lr_predictions = lr_cv_model.transform(test_df)

In [20]:
lr_predictions.select(

    "label",

    "prediction",

    "probability"

).show(10, truncate=False)

+-----+----------+-----------------------------------------------------------------------------------------------------+
|label|prediction|probability                                                                                          |
+-----+----------+-----------------------------------------------------------------------------------------------------+
|3.0  |0.0       |[0.5750282200369988,0.13520030083758183,0.10061531162122828,0.12062037026121183,0.06853579724297933] |
|3.0  |3.0       |[0.06675466425156408,0.1101735269035033,0.12324143749194903,0.6023645796585633,0.0974657916944201]   |
|3.0  |3.0       |[0.10204435893411248,0.04996216667628567,0.046788293458975,0.754651568084988,0.046553612845638706]   |
|3.0  |0.0       |[0.3226352045486764,0.16003158235931184,0.12217440181438058,0.3209612349338819,0.07419757634374936]  |
|3.0  |0.0       |[0.5929586284780003,0.18930173717920057,0.08246333871989557,0.08864315328114994,0.04663314234175351] |
|3.0  |0.0       |[0.45843125452

In [21]:
lr_predictions.printSchema()

root
 |-- rating: double (nullable = true)
 |-- text: string (nullable = true)
 |-- title: string (nullable = true)
 |-- review: string (nullable = true)
 |-- label: double (nullable = true)
 |-- words: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- filtered_words: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- rawFeatures: vector (nullable = true)
 |-- features: vector (nullable = true)
 |-- rawPrediction: vector (nullable = true)
 |-- probability: vector (nullable = true)
 |-- prediction: double (nullable = false)



In [22]:
print("Prediction Records:", lr_predictions.count())

Prediction Records: 96361


In [23]:
lr_predictions.groupBy("prediction").count().show()

+----------+-----+
|prediction|count|
+----------+-----+
|       1.0| 6083|
|       4.0|  287|
|       0.0|82370|
|       3.0| 4551|
|       2.0| 3070|
+----------+-----+



In [24]:
lr_predictions.cache()

lr_predictions.count()

print("Prediction dataset cached successfully.")

Prediction dataset cached successfully.


In [29]:
# Calculate all metrics including the missing F1 score
lr_accuracy = accuracy_eval.evaluate(lr_predictions)
lr_f1 = f1_eval.evaluate(lr_predictions)

print(f"Accuracy  : {lr_accuracy:.4f}")
print(f"Precision : {lr_precision:.4f}")
print(f"Recall    : {lr_recall:.4f}")
print(f"F1 Score  : {lr_f1:.4f}")

Accuracy  : 0.6964
Precision : 0.6504
Recall    : 0.6964
F1 Score  : 0.6342


## **Logistic Regression Performance**

In [30]:


print("=" * 60)
print("LOGISTIC REGRESSION PERFORMANCE")
print("=" * 60)

print(f"Accuracy      : {lr_accuracy:.4f}")
print(f"Precision     : {lr_precision:.4f}")
print(f"Recall        : {lr_recall:.4f}")
print(f"F1 Score      : {lr_f1:.4f}")
print(f"Training Time : {lr_training_time:.2f} seconds")

print("=" * 60)

LOGISTIC REGRESSION PERFORMANCE
Accuracy      : 0.6964
Precision     : 0.6504
Recall        : 0.6964
F1 Score      : 0.6342
Training Time : 120.14 seconds


# **Display Best Hyperparameters**

In [31]:

best_lr = lr_cv_model.bestModel

print("Best Logistic Regression Parameters")

print("-----------------------------------")

print("Max Iterations :", best_lr.getMaxIter())

print("Regularization Parameter :", best_lr.getRegParam())

print("ElasticNet Parameter :", best_lr.getElasticNetParam())

Best Logistic Regression Parameters
-----------------------------------
Max Iterations : 20
Regularization Parameter : 0.01
ElasticNet Parameter : 0.5


In [32]:
best_lr.write() \
    .overwrite() \
    .save("/content/drive/MyDrive/7006SCN/LR_Model")

print("Logistic Regression model saved successfully.")

Logistic Regression model saved successfully.


In [33]:
results.append({

    "Model": "Logistic Regression",

    "Accuracy": lr_accuracy,

    "Precision": lr_precision,

    "Recall": lr_recall,

    "F1 Score": lr_f1,

    "Training Time (Seconds)": lr_training_time

})

print("Results added successfully.")

Results added successfully.


In [34]:
print(results)

[{'Model': 'Logistic Regression', 'Accuracy': 0.6963605608077957, 'Precision': 0.6504381340007208, 'Recall': 0.6963605608077957, 'F1 Score': 0.634235332816163, 'Training Time (Seconds)': 120.14390850067139}]


In [35]:
import pandas as pd

pd.DataFrame(results)

,Model,Accuracy,Precision,Recall,F1 Score,Training Time (Seconds)
0,Logistic Regression,0.696361,0.650438,0.696361,0.634235,120.143909


In [36]:

lr_predictions.select(
    "label",
    "prediction",
    "probability"
).show(10, truncate=False)

+-----+----------+-----------------------------------------------------------------------------------------------------+
|label|prediction|probability                                                                                          |
+-----+----------+-----------------------------------------------------------------------------------------------------+
|3.0  |0.0       |[0.5750282200369988,0.13520030083758183,0.10061531162122828,0.12062037026121183,0.06853579724297933] |
|3.0  |3.0       |[0.06675466425156408,0.1101735269035033,0.12324143749194903,0.6023645796585633,0.0974657916944201]   |
|3.0  |3.0       |[0.10204435893411248,0.04996216667628567,0.046788293458975,0.754651568084988,0.046553612845638706]   |
|3.0  |0.0       |[0.3226352045486764,0.16003158235931184,0.12217440181438058,0.3209612349338819,0.07419757634374936]  |
|3.0  |0.0       |[0.5929586284780003,0.18930173717920057,0.08246333871989557,0.08864315328114994,0.04663314234175351] |
|3.0  |0.0       |[0.45843125452

In [37]:
lr_predictions.groupBy("prediction").count().show()

+----------+-----+
|prediction|count|
+----------+-----+
|       1.0| 6083|
|       4.0|  287|
|       0.0|82370|
|       3.0| 4551|
|       2.0| 3070|
+----------+-----+



In [38]:
correct_predictions = lr_predictions.filter(
    lr_predictions.label == lr_predictions.prediction
).count()

print("Correct Predictions:", correct_predictions)

Correct Predictions: 67102


In [39]:
incorrect_predictions = lr_predictions.filter(
    lr_predictions.label != lr_predictions.prediction
).count()

print("Incorrect Predictions:", incorrect_predictions)

Incorrect Predictions: 29259


In [40]:
print("=" * 60)
print("LOGISTIC REGRESSION MODEL COMPLETED SUCCESSFULLY")
print("=" * 60)

LOGISTIC REGRESSION MODEL COMPLETED SUCCESSFULLY


## **Reusable Model Training & Evaluation Function**
This function automates the training (with sampling), cross-validation, performance metrics calculation, and model saving for any Spark ML classifier.

In [44]:
def train_and_evaluate(model, param_grid, model_name, save_path):
    print(f"\n{'='*60}")
    print(f"STARTING: {model_name}")
    print(f"{'='*60}")

    start_time = time.time()

    try:
        # Step 1: Initialize CrossValidator
        cv = CrossValidator(
            estimator=model,
            estimatorParamMaps=param_grid,
            evaluator=accuracy_eval,
            numFolds=2,
            seed=SEED
        )

        # Step 2: Dynamic Sampling for Memory Safety
        train_count = train_df.count()
        sample_fraction = min(50000 / train_count, 1.0)
        train_sample = train_df.sample(withReplacement=False, fraction=sample_fraction, seed=SEED)
        print(f"Training {model_name} on {train_sample.count()} records (Sample fraction: {sample_fraction:.4f})...")

        # Step 3: Train Model
        cv_model = cv.fit(train_sample)
        end_time = time.time()
        duration = end_time - start_time
        print(f"Training completed in {duration:.2f} seconds.")

        # Step 4: Predictions and Evaluation
        predictions = cv_model.transform(test_df)

        acc = accuracy_eval.evaluate(predictions)
        prec = precision_eval.evaluate(predictions)
        rec = recall_eval.evaluate(predictions)
        f1 = f1_eval.evaluate(predictions)

        # Step 5: Save Best Model and CV Model
        best_model = cv_model.bestModel
        best_model.write().overwrite().save(save_path)
        cv_model.write().overwrite().save(save_path + "_CV")
        print(f"Models saved successfully to {save_path}")

        # Step 6: Store Results with Best Parameters
        results.append({
            "Model": model_name,
            "Accuracy": acc,
            "Precision": prec,
            "Recall": rec,
            "F1 Score": f1,
            "Training Time (Seconds)": duration,
            "Best Parameters": str(best_model.extractParamMap())
        })

        # Step 7: Print Performance Summary
        print(f"\n{model_name} Results:")
        print(f"- Accuracy  : {acc:.4f}")
        print(f"- Precision : {prec:.4f}")
        print(f"- Recall    : {rec:.4f}")
        print(f"- F1 Score  : {f1:.4f}")

        return cv_model

    except Exception as e:
        print(f"Error training {model_name}: {e}")
        return None

###  Linear Support Vector Machine (LinearSVC)


In [45]:
from pyspark.ml.classification import LinearSVC, OneVsRest

# Define the base Binary Model
lsvc = LinearSVC(featuresCol="features", labelCol="label", maxIter=10)

# Wrap it in OneVsRest for multi-class support
ovr = OneVsRest(classifier=lsvc, labelCol="label", featuresCol="features")

# Define ParamGrid
lsvc_paramGrid = (ParamGridBuilder()
                  .addGrid(lsvc.regParam, [0.01, 0.1])
                  .build())

# Run training
lsvc_model = train_and_evaluate(
    model=ovr,
    param_grid=lsvc_paramGrid,
    model_name="Linear SVM (OneVsRest)",
    save_path="/content/drive/MyDrive/7006SCN/LSVC_Model"
)


STARTING: Linear SVM (OneVsRest)
Training Linear SVM (OneVsRest) on 50110 records (Sample fraction: 0.1299)...
Training completed in 148.09 seconds.
Models saved successfully to /content/drive/MyDrive/7006SCN/LSVC_Model

Linear SVM (OneVsRest) Results:
- Accuracy  : 0.6994
- Precision : 0.6513
- Recall    : 0.6994
- F1 Score  : 0.6638


In [46]:
from pyspark.ml.classification import NaiveBayes

nb = NaiveBayes(featuresCol="features", labelCol="label")

nb_paramGrid = (ParamGridBuilder()
                .addGrid(nb.smoothing, [0.5, 1.0])
                .build())

nb_model = train_and_evaluate(
    model=nb,
    param_grid=nb_paramGrid,
    model_name="Naive Bayes",
    save_path="/content/drive/MyDrive/7006SCN/NB_Model"
)


STARTING: Naive Bayes
Training Naive Bayes on 50110 records (Sample fraction: 0.1299)...
Training completed in 74.60 seconds.
Models saved successfully to /content/drive/MyDrive/7006SCN/NB_Model

Naive Bayes Results:
- Accuracy  : 0.6248
- Precision : 0.6659
- Recall    : 0.6248
- F1 Score  : 0.6419


### ** Random Forest Classifier**
Training the final model using our optimized helper function.

In [52]:
from pyspark.ml.classification import RandomForestClassifier

# Define Model
rf = RandomForestClassifier(featuresCol="features", labelCol="label", seed=SEED)

# Define ParamGrid
rf_paramGrid = (ParamGridBuilder()
                .addGrid(rf.numTrees, [20, 50])
                .addGrid(rf.maxDepth, [5, 10])
                .build())

# Run training and evaluation
rf_model = train_and_evaluate(
    model=rf,
    param_grid=rf_paramGrid,
    model_name="Random Forest",
    save_path="/content/drive/MyDrive/7006SCN/RF_Model"
)


STARTING: Random Forest
Training Random Forest on 50110 records (Sample fraction: 0.1299)...
Training completed in 347.95 seconds.
Models saved successfully to /content/drive/MyDrive/7006SCN/RF_Model

Random Forest Results:
- Accuracy  : 0.6320
- Precision : 0.3994
- Recall    : 0.6320
- F1 Score  : 0.4895


## **Final Model Comparison Summary**

In [53]:
import pandas as pd

# Create final comparison table
comparison_df = pd.DataFrame(results).round(4)

# Sort by Accuracy
comparison_df = comparison_df.sort_values(by="Accuracy", ascending=False)

# Save to Drive
comparison_df.to_csv("/content/drive/MyDrive/7006SCN/model_comparison_final.csv", index=False)

print("Final Comparison Table saved to Drive.")
comparison_df

Final Comparison Table saved to Drive.


,Model,Accuracy,Precision,Recall,F1 Score,Training Time (Seconds),Best Parameters
1,Linear SVM (OneVsRest),0.6994,0.6513,0.6994,0.6638,148.0854,"{Param(parent='OneVsRestModel_f8dba0c5e9b7', n..."
0,Logistic Regression,0.6964,0.6504,0.6964,0.6342,120.1439,NaN
3,Random Forest,0.6320,0.3994,0.6320,0.4895,347.9453,{Param(parent='RandomForestClassifier_03fc4391...
2,Naive Bayes,0.6248,0.6659,0.6248,0.6419,74.6033,"{Param(parent='NaiveBayes_5cbc5bbfe07d', name=..."


In [54]:
import pandas as pd

comparison_df = pd.DataFrame(results)

comparison_df

,Model,Accuracy,Precision,Recall,F1 Score,Training Time (Seconds),Best Parameters
0,Logistic Regression,0.696361,0.650438,0.696361,0.634235,120.143909,NaN
1,Linear SVM (OneVsRest),0.699360,0.651270,0.699360,0.663795,148.085375,"{Param(parent='OneVsRestModel_f8dba0c5e9b7', n..."
2,Naive Bayes,0.624755,0.665942,0.624755,0.641872,74.603323,"{Param(parent='NaiveBayes_5cbc5bbfe07d', name=..."
3,Random Forest,0.631978,0.399396,0.631978,0.489462,347.945322,{Param(parent='RandomForestClassifier_03fc4391...


In [55]:
comparison_df = comparison_df.round(4)

comparison_df

,Model,Accuracy,Precision,Recall,F1 Score,Training Time (Seconds),Best Parameters
0,Logistic Regression,0.6964,0.6504,0.6964,0.6342,120.1439,NaN
1,Linear SVM (OneVsRest),0.6994,0.6513,0.6994,0.6638,148.0854,"{Param(parent='OneVsRestModel_f8dba0c5e9b7', n..."
2,Naive Bayes,0.6248,0.6659,0.6248,0.6419,74.6033,"{Param(parent='NaiveBayes_5cbc5bbfe07d', name=..."
3,Random Forest,0.6320,0.3994,0.6320,0.4895,347.9453,{Param(parent='RandomForestClassifier_03fc4391...


In [56]:
print("=" * 100)
print("MODEL COMPARISON SUMMARY")
print("=" * 100)

print(comparison_df)

MODEL COMPARISON SUMMARY
                    Model  Accuracy  Precision  Recall  F1 Score  \
0     Logistic Regression    0.6964     0.6504  0.6964    0.6342   
1  Linear SVM (OneVsRest)    0.6994     0.6513  0.6994    0.6638   
2             Naive Bayes    0.6248     0.6659  0.6248    0.6419   
3           Random Forest    0.6320     0.3994  0.6320    0.4895   

   Training Time (Seconds)                                    Best Parameters  
0                 120.1439                                                NaN  
1                 148.0854  {Param(parent='OneVsRestModel_f8dba0c5e9b7', n...  
2                  74.6033  {Param(parent='NaiveBayes_5cbc5bbfe07d', name=...  
3                 347.9453  {Param(parent='RandomForestClassifier_03fc4391...  


In [57]:
comparison_df = comparison_df.sort_values(
    by="Accuracy",
    ascending=False
)

comparison_df

,Model,Accuracy,Precision,Recall,F1 Score,Training Time (Seconds),Best Parameters
1,Linear SVM (OneVsRest),0.6994,0.6513,0.6994,0.6638,148.0854,"{Param(parent='OneVsRestModel_f8dba0c5e9b7', n..."
0,Logistic Regression,0.6964,0.6504,0.6964,0.6342,120.1439,NaN
3,Random Forest,0.6320,0.3994,0.6320,0.4895,347.9453,{Param(parent='RandomForestClassifier_03fc4391...
2,Naive Bayes,0.6248,0.6659,0.6248,0.6419,74.6033,"{Param(parent='NaiveBayes_5cbc5bbfe07d', name=..."


In [58]:
best_model = comparison_df.iloc[0]

print("Best Model")

print(best_model)

Best Model
Model                                                 Linear SVM (OneVsRest)
Accuracy                                                              0.6994
Precision                                                             0.6513
Recall                                                                0.6994
F1 Score                                                              0.6638
Training Time (Seconds)                                             148.0854
Best Parameters            {Param(parent='OneVsRestModel_f8dba0c5e9b7', n...
Name: 1, dtype: object


In [58]:
comparison_df.to_csv(
    "/content/drive/MyDrive/7006SCN/model_comparison.csv",
    index=False
)

print("Comparison table saved.")

In [59]:
comparison_df.head()

,Model,Accuracy,Precision,Recall,F1 Score,Training Time (Seconds),Best Parameters
1,Linear SVM (OneVsRest),0.6994,0.6513,0.6994,0.6638,148.0854,"{Param(parent='OneVsRestModel_f8dba0c5e9b7', n..."
0,Logistic Regression,0.6964,0.6504,0.6964,0.6342,120.1439,NaN
3,Random Forest,0.6320,0.3994,0.6320,0.4895,347.9453,{Param(parent='RandomForestClassifier_03fc4391...
2,Naive Bayes,0.6248,0.6659,0.6248,0.6419,74.6033,"{Param(parent='NaiveBayes_5cbc5bbfe07d', name=..."


In [61]:
import os

files = os.listdir("/content/drive/MyDrive/7006SCN")

for file in files:
    print(file)

Pipeline_Task2
processed_reviews
LR_Model
LSVC_Model
LSVC_Model_CV
NB_Model
NB_Model_CV
RF_Model
RF_Model_CV
model_comparison_final.csv


In [62]:
print("=" * 80)
print("TASK 3 COMPLETED SUCCESSFULLY")
print("=" * 80)

print("Models Trained")

print("1. Logistic Regression")

print("2. Linear SVM")

print("3. Naive Bayes")

print("4. Random Forest")

print()

print("Comparison table created")

print("Models saved")

print("Results exported")

TASK 3 COMPLETED SUCCESSFULLY
Models Trained
1. Logistic Regression
2. Linear SVM
3. Naive Bayes
4. Random Forest

Comparison table created
Models saved
Results exported
